# Machine Learning Coding Interviews

Run all cells in a fresh Python3.11+ kernel. CPU/NumPy only; no API keys or data downloads. The user approved replacing the source K-means section's mistaken KNN implementation with actual tested K-means. All original blocks, including that erroneous class, remain in the appendix. Other maintenance: stable sigmoid, stable nearest-neighbor ordering and first-nearest tie voting. Source defines algorithms but no dataset or training example; explicit small deterministic examples are added. Logistic regression executes its original100000-iteration default, not a reduced smoke test. Independent least-squares/scalar-neighbor/exhaustive-clustering checks validate results. Original prose's other correctness/terminology findings stay in the private review rather than added public editorial notes.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable,"-m","pip","install","--quiet","--disable-pip-version-check","numpy==2.4.6"],check=True)
import numpy as np
from collections import Counter
np.set_printoptions(precision=6,suppress=True)
rng=np.random.default_rng(42)
print("Python:",sys.version.split()[0],"| NumPy:",np.__version__)

## Example 0

In [ ]:
class LinearRegression() :
      
    def __init__( self, learning_rate, iterations ) :
          
        self.learning_rate = learning_rate
          
        self.iterations = iterations
          
    # Function for model training
              
    def fit( self, X, Y ) :
          
        # no_of_training_examples, no_of_features
          
        self.m, self.n = X.shape
          
        # weight initialization
          
        self.W = np.zeros( self.n )
          
        self.b = 0
          
        self.X = X
          
        self.Y = Y
          
          
        # gradient descent learning
                  
        for i in range( self.iterations ) :
              
            self.update_weights()
              
        return self
      
    # Helper function to update weights in gradient descent
      
    def update_weights( self ) :
             
        Y_pred = self.predict( self.X )
          
        # calculate gradients  
      
        dW = - ( 2 * ( self.X.T ).dot( self.Y - Y_pred )  ) / self.m
       
        db = - 2 * np.sum( self.Y - Y_pred ) / self.m 
          
        # update weights
      
        self.W = self.W - self.learning_rate * dW
      
        self.b = self.b - self.learning_rate * db
          
        return self
      
    # Hypothetical function  h( x ) 
      
    def predict( self, X ) :
      
        return X.dot( self.W ) + self.b

X=rng.normal(size=(100,2))
y=X@np.array([2.,-3.])+1.5
linear=LinearRegression(learning_rate=0.1,iterations=3000).fit(X,y)
print("Linear weights:",linear.W,"| bias:",f"{linear.b:.6f}")
print("Training MSE:",f"{np.mean((linear.predict(X)-y)**2):.12f}")

## Example 1

In [ ]:
import numpy as np

class LogisticRegression:
    def __init__(self, learning_rate=0.01, num_iterations=100000, fit_intercept=True):
        self.learning_rate = learning_rate
        self.num_iterations = num_iterations
        self.fit_intercept = fit_intercept
    
    def sigmoid(self, z):
        return np.exp(-np.logaddexp(0, -z))
    
    def fit(self, X, y):
        if self.fit_intercept:
            X = np.c_[np.ones((X.shape[0], 1)), X]
        
        self.theta = np.zeros(X.shape[1])
        
        for i in range(self.num_iterations):
            z = np.dot(X, self.theta)
            h = self.sigmoid(z)
            gradient = np.dot(X.T, (h - y)) / y.size
            self.theta -= self.learning_rate * gradient
    
    def predict_prob(self, X):
        if self.fit_intercept:
            X = np.c_[np.ones((X.shape[0], 1)), X]
    
        return self.sigmoid(np.dot(X, self.theta))
    
    def predict(self, X, threshold=0.5):
        return self.predict_prob(X) >= threshold

logistic_X=np.array([[-2.,-1.],[-1.,-2.],[-1.,-1.],[1.,1.],[1.,2.],[2.,1.]])
logistic_y=np.array([0,0,0,1,1,1])
logistic=LogisticRegression()
logistic.fit(logistic_X,logistic_y)
print("Logistic iterations:",logistic.num_iterations)
print("Probabilities:",logistic.predict_prob(logistic_X))
print("Classes:",logistic.predict(logistic_X).astype(int))

## Example 2

In [ ]:
import numpy as np

def euclidean_distance(x, y):
    """Calculates the Euclidean distance between two instances"""
    return np.sqrt(np.sum((x - y) ** 2))

def k_nearest_neighbors(X_train, y_train, x_test, k):
    """
    Finds the k nearest neighbors of a test instance in the training dataset
    and returns their indices and distances
    """
    distances = [euclidean_distance(x_test, x) for x in X_train]
    nearest_neighbors = np.argsort(distances, kind="stable")[:k]
    return nearest_neighbors, [distances[i] for i in nearest_neighbors]

def k_nearest_neighbors_classification(X_train, y_train, x_test, k):
    """
    Finds the k nearest neighbors of a test instance in the training dataset
    and returns the class that is most common among them
    """
    nearest_neighbors, _ = k_nearest_neighbors(X_train, y_train, x_test, k)
    classes = [y_train[i] for i in nearest_neighbors]
    return Counter(classes).most_common(1)[0][0]

def k_nearest_neighbors_regression(X_train, y_train, x_test, k):
    """
    Finds the k nearest neighbors of a test instance in the training dataset
    and returns the average value among them
    """
    nearest_neighbors, _ = k_nearest_neighbors(X_train, y_train, x_test, k)
    return np.mean([y_train[i] for i in nearest_neighbors])

knn_X=np.array([[0.,0.],[1.,0.],[0.,1.],[4.,4.],[5.,4.],[4.,5.]])
knn_y=np.array([0,0,0,1,1,1])
query=np.array([0.2,0.1])
indices,distances=k_nearest_neighbors(knn_X,knn_y,query,3)
print("Nearest indices:",indices,"| distances:",np.round(distances,6))
print("Predicted class:",k_nearest_neighbors_classification(knn_X,knn_y,query,3))
print("Regression mean:",k_nearest_neighbors_regression(knn_X,np.arange(6.),query,3))

## Example 5

In [ ]:
class KMeans:
    def __init__(self,k=3,max_iterations=300,n_init=10,random_state=42):
        self.k,self.max_iterations,self.n_init,self.random_state=k,max_iterations,n_init,random_state

    def fit(self,X):
        X=np.asarray(X,dtype=float)
        if X.ndim!=2 or not np.isfinite(X).all() or not 1<=self.k<=len(X):raise ValueError("Need finite 2D data and 1 <= k <= number of rows")
        if self.n_init<1 or self.max_iterations<1:raise ValueError("Iteration and restart counts must be positive")
        random=np.random.default_rng(self.random_state)
        best=None
        for _ in range(self.n_init):
            centers=X[random.choice(len(X),self.k,replace=False)].copy()
            for iteration in range(self.max_iterations):
                labels=((X[:,None,:]-centers[None,:,:])**2).sum(axis=2).argmin(axis=1)
                updated=np.array([X[labels==j].mean(axis=0) if np.any(labels==j) else centers[j] for j in range(self.k)])
                converged=np.allclose(updated,centers,atol=1e-10,rtol=0)
                centers=updated
                if converged:break
            labels=((X[:,None,:]-centers[None,:,:])**2).sum(axis=2).argmin(axis=1)
            inertia=float(((X-centers[labels])**2).sum())
            if best is None or inertia<best[0]:best=(inertia,centers.copy(),labels.copy(),iteration+1)
        self.inertia_,self.cluster_centers_,self.labels_,self.n_iter_=best
        return self

    def predict(self,X):
        X=np.asarray(X,dtype=float)
        return ((X[:,None,:]-self.cluster_centers_[None,:,:])**2).sum(axis=2).argmin(axis=1)

cluster_X=np.array([[0.,0.],[0.,1.],[1.,0.],[10.,10.],[10.,11.],[11.,10.]])
kmeans=KMeans(k=2).fit(cluster_X)
print("Centroids:")
print(kmeans.cluster_centers_)
print("Cluster labels:",kmeans.labels_)
print("Within-cluster sum of squares:",f"{kmeans.inertia_:.6f}")

## Independent regression checks

In [ ]:
from itertools import product
augmented=np.c_[X,np.ones(len(X))]
least_squares=np.linalg.lstsq(augmented,y,rcond=None)[0]
np.testing.assert_allclose(np.r_[linear.W,linear.b],least_squares,atol=1e-10)
assert np.mean((linear.predict(X)-y)**2)<1e-20
assert logistic.num_iterations==100000
np.testing.assert_array_equal(logistic.predict(logistic_X),logistic_y.astype(bool))
np.testing.assert_allclose(logistic.sigmoid(np.array([-1000.,0.,1000.])),[0.,0.5,1.])
no_intercept=LogisticRegression(num_iterations=100000,fit_intercept=False)
no_intercept.fit(logistic_X,logistic_y)
np.testing.assert_array_equal(no_intercept.predict(logistic_X),logistic_y.astype(bool))
for point in [np.array([0.2,0.1]),np.array([4.5,4.2]),np.array([1.,2.])]:
    scalar=np.array([sum((float(a)-float(b))**2 for a,b in zip(point,row))**0.5 for row in knn_X])
    order=sorted(range(len(scalar)),key=lambda i:(scalar[i],i))[:3]
    indices,distances=k_nearest_neighbors(knn_X,knn_y,point,3)
    np.testing.assert_array_equal(indices,order)
    np.testing.assert_allclose(distances,scalar[order])
    assert k_nearest_neighbors_classification(knn_X,knn_y,point,3)==Counter(knn_y[order]).most_common(1)[0][0]
    np.testing.assert_allclose(k_nearest_neighbors_regression(knn_X,np.arange(6.),point,3),np.mean(order))
tie_X=np.array([[-1.],[1.]])
assert k_nearest_neighbors_classification(tie_X,np.array([9,2]),np.array([0.]),2)==9
minimum=float("inf")
for assignment in product(range(2),repeat=len(cluster_X)):
    labels=np.array(assignment)
    if len(set(assignment))!=2:continue
    centers=np.array([cluster_X[labels==j].mean(axis=0) for j in range(2)])
    minimum=min(minimum,float(((cluster_X-centers[labels])**2).sum()))
np.testing.assert_allclose(kmeans.inertia_,minimum,atol=1e-12)
np.testing.assert_array_equal(kmeans.predict(cluster_X),kmeans.labels_)
for j in range(2):np.testing.assert_allclose(kmeans.cluster_centers_[j],cluster_X[kmeans.labels_==j].mean(axis=0))
repeat=KMeans(k=2).fit(cluster_X)
np.testing.assert_array_equal(repeat.labels_,kmeans.labels_)
duplicate=KMeans(k=3).fit(np.ones((6,2)))
assert duplicate.inertia_==0 and np.isfinite(duplicate.cluster_centers_).all()
try:KMeans(k=7).fit(cluster_X)
except ValueError:pass
else:raise AssertionError("Invalid k accepted")
print("Linear regression matches independent least squares.")
print("Both logistic intercept modes completed all100000 updates; extreme sigmoid finite.")
print("KNN matches scalar distances/votes/means with a deterministic tie rule.")
print("K-means matches exhaustive two-cluster minimum, centroid means and nearest-center predictions; empty clusters and invalid k checked.")

## Original source appendix

Full original blocks from raw source SHA-256 0ad39c0605811ff8cf6cdf43d34db9119cf60509904a265643bc0bc58621f397. Preserved verbatim as nonexecuting historical material, including mistakes/legacy setup alternatives.

### Original block 1

```text
class LinearRegression() :
      
    def __init__( self, learning_rate, iterations ) :
          
        self.learning_rate = learning_rate
          
        self.iterations = iterations
          
    # Function for model training
              
    def fit( self, X, Y ) :
          
        # no_of_training_examples, no_of_features
          
        self.m, self.n = X.shape
          
        # weight initialization
          
        self.W = np.zeros( self.n )
          
        self.b = 0
          
        self.X = X
          
        self.Y = Y
          
          
        # gradient descent learning
                  
        for i in range( self.iterations ) :
              
            self.update_weights()
              
        return self
      
    # Helper function to update weights in gradient descent
      
    def update_weights( self ) :
             
        Y_pred = self.predict( self.X )
          
        # calculate gradients  
      
        dW = - ( 2 * ( self.X.T ).dot( self.Y - Y_pred )  ) / self.m
       
        db = - 2 * np.sum( self.Y - Y_pred ) / self.m 
          
        # update weights
      
        self.W = self.W - self.learning_rate * dW
      
        self.b = self.b - self.learning_rate * db
          
        return self
      
    # Hypothetical function  h( x ) 
      
    def predict( self, X ) :
      
        return X.dot( self.W ) + self.b
```

### Original block 2

```text
import numpy as np

class LogisticRegression:
    def __init__(self, learning_rate=0.01, num_iterations=100000, fit_intercept=True):
        self.learning_rate = learning_rate
        self.num_iterations = num_iterations
        self.fit_intercept = fit_intercept
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        if self.fit_intercept:
            X = np.c_[np.ones((X.shape[0], 1)), X]
        
        self.theta = np.zeros(X.shape[1])
        
        for i in range(self.num_iterations):
            z = np.dot(X, self.theta)
            h = self.sigmoid(z)
            gradient = np.dot(X.T, (h - y)) / y.size
            self.theta -= self.learning_rate * gradient
    
    def predict_prob(self, X):
        if self.fit_intercept:
            X = np.c_[np.ones((X.shape[0], 1)), X]
    
        return self.sigmoid(np.dot(X, self.theta))
    
    def predict(self, X, threshold=0.5):
        return self.predict_prob(X) >= threshold
```

### Original block 3

```text
import numpy as np

def euclidean_distance(x, y):
    """Calculates the Euclidean distance between two instances"""
    return np.sqrt(np.sum((x - y) ** 2))

def k_nearest_neighbors(X_train, y_train, x_test, k):
    """
    Finds the k nearest neighbors of a test instance in the training dataset
    and returns their indices and distances
    """
    distances = [euclidean_distance(x_test, x) for x in X_train]
    nearest_neighbors = np.argsort(distances)[:k]
    return nearest_neighbors, [distances[i] for i in nearest_neighbors]

def k_nearest_neighbors_classification(X_train, y_train, x_test, k):
    """
    Finds the k nearest neighbors of a test instance in the training dataset
    and returns the class that is most common among them
    """
    nearest_neighbors, _ = k_nearest_neighbors(X_train, y_train, x_test, k)
    classes = [y_train[i] for i in nearest_neighbors]
    return max(set(classes), key = classes.count)

def k_nearest_neighbors_regression(X_train, y_train, x_test, k):
    """
    Finds the k nearest neighbors of a test instance in the training dataset
    and returns the average value among them
    """
    nearest_neighbors, _ = k_nearest_neighbors(X_train, y_train, x_test, k)
    return np.mean([y_train[i] for i in nearest_neighbors])
```

### Original block 4

```text
c(i) = argmin(||x_i - c_j||^2) for j = 1, 2, ..., k
```

### Original block 5

```text
c_j = (1/n_j) * sum(x_i for i in c_j) for j = 1, 2, ..., k
```

### Original block 6

```text
import numpy as np

def euclidean_distance(x1, x2):
    """
    Calculates the euclidean distance between two data points
    """
    return np.sqrt(np.sum((x1 - x2)**2))

class KNN:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        """
        X : array-like, shape (n_samples, n_features)
            Training data
        y : array-like, shape (n_samples,)
            Target values
        """
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        """
        X : array-like, shape (n_samples, n_features)
            Test data
        """
        y_pred = []
        for x in X:
            distances = [euclidean_distance(x, x_train) for x_train in self.X_train]
            # find k nearest neighbors
            k_neighbors = np.argsort(distances)[:self.k]
            # predict the most common class among k nearest neighbors
            k_neighbors_labels = [self.y_train[i] for i in k_neighbors]
            y_pred.append(max(set(k_neighbors_labels), key=k_neighbors_labels.count))
        return np.array(y_pred)
```